# Task 2.1 — Albero di decisione costruito a mano

Costruiamo un albero di decisione sul file `manuale.csv` (14 campioni bilanciati)
**senza usare il fit automatico**: feature e soglie di split le scegliamo noi
calcolando entropia e information gain (algoritmo ID3 per feature continue).

Documentazione completa con teoria e motivazioni: `docs/02.1_albero_decisionale.md`

## 1. Caricamento dei 14 campioni

Carichiamo `manuale.csv` e teniamo solo le colonne utili: le 4 feature e la LABEL.
Gli identificatori (ACTIONID, USERID, TARGETID) e TIMESTAMP non sono caratteristiche
generalizzabili e non vanno usati come feature.

In [1]:
# Importiamo pandas per leggere e manipolare tabelle
import pandas as pd

# Carichiamo manuale.csv: stavolta è un CSV "vero" (separato da virgole),
# quindi non serve sep="\t" come per i .tsv originali
manuale = pd.read_csv("../data/manuale.csv")

# Teniamo solo le colonne che ci servono per l'albero:
# le 4 feature (gli input) e la LABEL (la risposta da prevedere).
# Doppia parentesi quadra perché passiamo una LISTA di nomi di colonne.
df = manuale[["FEATURE0", "FEATURE1", "FEATURE2", "FEATURE3", "LABEL"]]

# Mostriamo TUTTE le righe, le stampa tutte:
df

,FEATURE0,FEATURE1,FEATURE2,FEATURE3,LABEL
0,-0.319991,-0.435701,0.106784,-0.067309,0
1,-0.319991,-0.435701,0.106784,-0.067309,0
2,-0.319991,-0.435701,0.106784,-0.067309,1
3,2.376173,-0.435701,-0.394237,-0.067309,0
4,2.376173,-0.435701,-0.394237,-0.067309,1
5,-0.319991,-0.435701,0.106784,-0.067309,0
6,-0.319991,-0.435701,0.106784,-0.067309,1
7,-0.319991,2.108722,-0.394237,-0.067309,1
8,-0.319991,2.108722,-0.394237,-0.067309,0
9,-0.319991,-0.435701,0.607805,-0.067309,1


## 2. Funzioni di supporto: entropia e information gain

- Entropia(S) = −p0·log2(p0) − p1·log2(p1)  (con la convenzione 0·log2(0) = 0)
- IG = Entropia(padre) − media pesata delle entropie dei due figli

Scriviamo due piccole funzioni per non ripetere i calcoli.

In [2]:
# math.log2 = logaritmo in base 2.
from math import log2

def entropia(labels):
    """Calcola l'entropia di un gruppo di campioni, date le loro label (0/1)."""
    n = len(labels)              # quanti campioni ci sono nel gruppo
    if n == 0:
        return 0                 # gruppo vuoto: nessun disordine per convenzione

    p1 = sum(labels) / n         # frazione di 1 (le label sono 0/1, la somma conta gli 1)
    p0 = 1 - p1                  # frazione di 0 (il resto)

    e = 0
    # convenzione: 0 * log2(0) = 0, quindi sommiamo un termine solo se p > 0
    # (log2(0) non esiste e darebbe errore)
    if p0 > 0:
        e -= p0 * log2(p0)
    if p1 > 0:
        e -= p1 * log2(p1)
    return e


def info_gain(labels_padre, labels_sx, labels_dx):
    """Information Gain di uno split che divide il padre nei due rami sx e dx."""
    n = len(labels_padre)
    peso_sx = len(labels_sx) / n   # frazione di campioni finita nel ramo sinistro
    peso_dx = len(labels_dx) / n   # frazione nel ramo destro

    # IG = entropia prima dello split - media pesata delle entropie dopo
    return entropia(labels_padre) - (peso_sx * entropia(labels_sx) +
                                     peso_dx * entropia(labels_dx))


# --- mini-test per verificare che funzionino ---
print(entropia([0, 0, 0]))        # atteso: 0.0   (gruppo puro)
print(entropia([0, 1]))           # atteso: 1.0   (50/50, massimo disordine)
print(entropia([1, 1, 1, 0]))     # atteso: ~0.811 (3 contro 1)

0.0
1.0
0.8112781244591328


## 3. Entropia della radice

Il nodo radice contiene tutti e 14 i campioni (7 dropout, 7 no):
l'entropia attesa è esattamente **1.0** (massima incertezza).

In [3]:
# L'entropia della radice: passiamo alla funzione la colonna LABEL intera
# (tutti e 14 i campioni). La convertiamo in lista per semplicità.
e_radice = entropia(list(df["LABEL"]))
print("Entropia della radice:", e_radice)

Entropia della radice: 1.0


## 4. Ricerca dello split migliore per la radice

Per ogni feature:
1. ordiniamo i 14 campioni per quella feature;
2. le soglie candidate sono i punti medi tra valori consecutivi di classe diversa;
3. per ogni candidata calcoliamo l'IG e teniamo la migliore.

Alla fine confrontiamo le 4 migliori (una per feature) e scegliamo la radice.

Almeno un calcolo di IG va rifatto **a mano su carta** e riportato nella documentazione.

In [10]:
# Per ogni feature stampiamo i valori ordinati con la label affiancata:
# serve a vedere DOVE la label cambia scorrendo i valori, cioè dove
# hanno senso le soglie candidate (come nell'esempio ore_studio/esito)
for feat in ["FEATURE0", "FEATURE1", "FEATURE2", "FEATURE3"]:
    print(feat)
    # ordiniamo le 14 righe per il valore di questa feature
    ordinato = df.sort_values(feat)
    # stampiamo coppie (valore, label) in ordine crescente
    for _, riga in ordinato.iterrows():
        print(f"  {riga[feat]:8.4f}   label={int(riga['LABEL'])}")

def migliori_split(dati, feature):
    """Prova tutte le soglie candidate di una feature e restituisce
    la lista (soglia, IG) ordinata dal gain più alto."""
    ordinato = dati.sort_values(feature).reset_index(drop=True)
    valori = list(ordinato[feature])
    labels = list(ordinato["LABEL"])
    risultati = []

    # scorriamo le coppie di campioni consecutivi
    for i in range(len(valori) - 1):
        # soglia candidata SOLO dove la label cambia (altrove l'IG non migliora)
        # e solo se i due valori sono diversi (con valori uguali non si può tagliare)
        if labels[i] != labels[i + 1] and valori[i] != valori[i + 1]:
            soglia = (valori[i] + valori[i + 1]) / 2   # punto medio

            # dividiamo le label nei due rami secondo la domanda "feature <= soglia?"
            sx = [l for v, l in zip(valori, labels) if v <= soglia]  # ramo "sì"
            dx = [l for v, l in zip(valori, labels) if v > soglia]   # ramo "no"

            risultati.append((soglia, info_gain(labels, sx, dx)))

    # ordiniamo dal gain più alto al più basso
    return sorted(risultati, key=lambda x: x[1], reverse=True)


# Proviamo tutte e 4 le feature e stampiamo le loro candidate migliori
for feat in ["FEATURE0", "FEATURE1", "FEATURE2", "FEATURE3"]:
    candidate = migliori_split(df, feat)
    print("=" * 60)
    print(feat, "-", len(candidate), "soglie candidate")
    for soglia, ig in candidate[:3]:   # mostriamo solo le 3 migliori per feature
        print(f"   soglia {soglia:8.4f}  ->  IG = {ig:.4f}")

FEATURE0
   -0.3200   label=0
   -0.3200   label=0
   -0.3200   label=1
   -0.3200   label=0
   -0.3200   label=1
   -0.3200   label=1
   -0.3200   label=1
   -0.3200   label=0
   -0.3200   label=1
   -0.3200   label=1
   -0.3200   label=0
    2.3762   label=0
    2.3762   label=1
    2.3762   label=0
FEATURE1
   -0.4357   label=0
   -0.4357   label=0
   -0.4357   label=1
   -0.4357   label=0
   -0.4357   label=1
   -0.4357   label=0
   -0.4357   label=1
   -0.4357   label=1
   -0.4357   label=1
   -0.4357   label=0
    2.1087   label=1
    2.1087   label=0
    2.1087   label=0
    2.1087   label=1
FEATURE2
   -0.3942   label=0
   -0.3942   label=1
   -0.3942   label=1
   -0.3942   label=0
   -0.3942   label=1
   -0.3942   label=0
   -0.3942   label=0
    0.1068   label=0
    0.1068   label=1
    0.1068   label=0
    0.1068   label=0
    0.1068   label=1
    0.1068   label=1
    0.6078   label=1
FEATURE3
   -0.0673   label=0
   -0.0673   label=0
   -0.0673   label=1
   -0.0673   label=

## 5. Costruzione dell'albero (profondità max 2-3)

Scelta la radice, guardiamo i due rami: se uno è impuro ripetiamo la ricerca
dello split **solo sui campioni di quel ramo**. Ci fermiamo quando i rami sono
puri o alla profondità massima, assegnando alle foglie la classe di maggioranza.

Qui sotto annotiamo l'albero finale come regole esplicite, ad esempio:
```
SE FEATUREx <= t1:
    SE FEATUREy <= t2:  → classe A
    ALTRIMENTI:         → classe B
ALTRIMENTI:             → classe C
```

In [ ]:
# TODO (Fabio): ripeti la ricerca dello split sui rami impuri della radice
# (filtra il dataframe con la condizione della radice e riusa il codice del punto 4).


## 6. Implementazione dell'albero in Python

L'albero è già "addestrato" (le soglie le abbiamo scelte noi): la sua
implementazione è una semplice catena di if/else.

In [ ]:
# TODO (Fabio): scrivi la funzione predici(riga) che traduce le regole del punto 5
# in if/else e restituisce 0 o 1. Poi applicala a tutte le 14 righe
# (df.apply(predici, axis=1)) salvando il risultato in una colonna PREDETTA.


## 7. Valutazione sulle 14 osservazioni

Confrontiamo PREDETTA con LABEL:
- **Confusion matrix**: TP, TN, FP, FN
- **Accuracy** = (TP+TN)/14, **Precision** = TP/(TP+FP), **Recall** = TP/(TP+FN),
  **F1** = 2·P·R/(P+R)

Nota: valutare sugli stessi campioni usati per costruire l'albero dà una stima
ottimistica — lo discutiamo nell'analisi critica e lo verificheremo allo Step 4.

In [ ]:
# TODO (Fabio): calcola TP, TN, FP, FN confrontando le colonne LABEL e PREDETTA,
# poi accuracy, precision, recall e F1. Puoi farlo con semplici confronti booleani
# (es. ((df.LABEL==1) & (df.PREDETTA==1)).sum() per i TP).


## 8. Analisi critica

*(da completare coi risultati alla mano — traccia in `docs/02.1_albero_decisionale.md`,
sezione 5: stima ottimistica, instabilità con 14 campioni, overfitting delle foglie
piccole, feature anonime non interpretabili, rimando allo Step 4)*